# Phase 4: Production Processing - 30K Images

**Objectif:** Processer l'ensemble du dataset unifié (30,513 images)

**Stratégie:**
- Traitement parallèle par classe (5 workers)
- Checkpointing automatique tous les 1,000 images
- Sauvegarde progressive (images + métadonnées)
- Recovery automatique en cas d'interruption

**Inputs:**
- `../data/unified_dataset_manifest.csv` (30,513 images)
- Pipeline validé (96% primary, 4% fallback)

**Outputs:**
- `outputs/processed/` - Images traitées (448×448 PNG)
- `outputs/processed/metadata/` - Métadonnées JSON par image
- `outputs/processed/checkpoints/` - Checkpoints par classe
- `outputs/processed/production_summary.json` - Statistiques finales

**Temps estimé:** ~2-3 heures (RTX 5090, 266 ms/image)

## 1. Setup & Imports

In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import time
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import cv2
import xml.etree.ElementTree as ET
from datetime import datetime
import multiprocessing as mp
from functools import partial

# Torch imports
import torch

# Transformers for GroundingDINO
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

# SAM
from segment_anything import sam_model_registry, SamPredictor

print("✓ Imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print(f"CPU cores available: {mp.cpu_count()}")

✓ Imports successful
PyTorch version: 2.9.1+cu128
CUDA available: True
CUDA device: NVIDIA GeForce RTX 5090
CPU cores available: 24


## 2. Configuration

In [2]:
# Paths
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / 'data'
OUTPUT_DIR = BASE_DIR / 'outputs' / 'processed'

# Create output structure
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'images').mkdir(exist_ok=True)
(OUTPUT_DIR / 'metadata').mkdir(exist_ok=True)
(OUTPUT_DIR / 'checkpoints').mkdir(exist_ok=True)

# Create class subdirectories
CLASS_NAMES = {
    0: 'CBB',
    1: 'CBSD',
    2: 'CGM',
    3: 'CMD',
    4: 'Healthy'
}

for label, name in CLASS_NAMES.items():
    (OUTPUT_DIR / 'images' / f"{label}_{name}").mkdir(exist_ok=True)
    (OUTPUT_DIR / 'metadata' / f"{label}_{name}").mkdir(exist_ok=True)

# Unified manifest
MANIFEST_PATH = DATA_DIR / 'unified_dataset_manifest.csv'

# GroundingDINO model from Hugging Face
GROUNDING_DINO_MODEL_ID = "IDEA-Research/grounding-dino-tiny"

# SAM model
SAM_MODEL_TYPE = "vit_h"
SAM_CHECKPOINT = BASE_DIR / 'models' / 'sam_vit_h_4b8939.pth'

# Processing params
TARGET_SIZE = (448, 448)
PADDING_RATIO = 0.1

# Detection params
TEXT_THRESHOLD = 0.25
PROMPTS = ["cassava leaf", "cassava plant leaf", "green leaf"]

# Class-specific adaptive thresholds (validated in Phase 3)
CLASS_THRESHOLDS = {
    0: {'box_threshold': 0.30, 'min_area_ratio': 0.025},
    1: {'box_threshold': 0.30, 'min_area_ratio': 0.025},
    2: {'box_threshold': 0.25, 'min_area_ratio': 0.015},
    3: {'box_threshold': 0.30, 'min_area_ratio': 0.025},
    4: {'box_threshold': 0.28, 'min_area_ratio': 0.020},
}

# Fallback configuration
FALLBACK_ENABLED = True
FALLBACK_METHOD = 'green_semantic'
GREEN_PERCENTILE_THRESHOLD = 60

# Production processing params
CHECKPOINT_INTERVAL = 1000  # Save checkpoint every N images
NUM_WORKERS = 5  # Process 5 classes in parallel
BATCH_SIZE = 100  # Process images in batches for better memory management

print("✓ Configuration loaded")
print(f"\nOutput structure:")
print(f"  Images: {OUTPUT_DIR / 'images'}")
print(f"  Metadata: {OUTPUT_DIR / 'metadata'}")
print(f"  Checkpoints: {OUTPUT_DIR / 'checkpoints'}")
print(f"\nProcessing config:")
print(f"  Workers: {NUM_WORKERS}")
print(f"  Checkpoint interval: {CHECKPOINT_INTERVAL} images")
print(f"  Batch size: {BATCH_SIZE}")

✓ Configuration loaded

Output structure:
  Images: /home/hounfodjidagba/learning/cassava/outputs/processed/images
  Metadata: /home/hounfodjidagba/learning/cassava/outputs/processed/metadata
  Checkpoints: /home/hounfodjidagba/learning/cassava/outputs/processed/checkpoints

Processing config:
  Workers: 5
  Checkpoint interval: 1000 images
  Batch size: 100


## 3. Load Manifest

In [3]:
print("Loading unified dataset manifest...")
manifest_df = pd.read_csv(MANIFEST_PATH)

print(f"✓ Manifest loaded: {len(manifest_df):,} images")
print(f"\nDistribution by class:")
class_counts = manifest_df['label'].value_counts().sort_index()
for label, count in class_counts.items():
    print(f"  {label} - {CLASS_NAMES[label]:10s}: {count:,} images")

print(f"\nTotal to process: {len(manifest_df):,} images")
print(f"Estimated time: {len(manifest_df) * 0.266 / 3600:.1f} hours @ 266 ms/image")

Loading unified dataset manifest...
✓ Manifest loaded: 30,513 images

Distribution by class:
  0 - CBB       : 1,087 images
  1 - CBSD      : 5,186 images
  2 - CGM       : 2,386 images
  3 - CMD       : 18,158 images
  4 - Healthy   : 3,696 images

Total to process: 30,513 images
Estimated time: 2.3 hours @ 266 ms/image


## 4. Load Models (Single Instance for All Workers)

In [4]:
print("="*70)
print("LOADING MODELS")
print("="*70)

# Load GroundingDINO from Hugging Face
print("\nLoading GroundingDINO...")
grounding_dino_processor = AutoProcessor.from_pretrained(GROUNDING_DINO_MODEL_ID)
grounding_dino_model = AutoModelForZeroShotObjectDetection.from_pretrained(GROUNDING_DINO_MODEL_ID)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
grounding_dino_model.to(device)
grounding_dino_model.eval()
print(f"✓ GroundingDINO loaded on {device}")

# Load SAM model
print("\nLoading SAM...")
sam = sam_model_registry[SAM_MODEL_TYPE](checkpoint=str(SAM_CHECKPOINT))
sam.to(device)
sam_predictor = SamPredictor(sam)
print(f"✓ SAM loaded on {device}")

print("\n" + "="*70)
print("MODELS READY FOR PRODUCTION")
print("="*70)

LOADING MODELS

Loading GroundingDINO...
✓ GroundingDINO loaded on cuda

Loading SAM...
✓ SAM loaded on cuda

MODELS READY FOR PRODUCTION


## 5. Pipeline Functions

Copie des fonctions validées du notebook 04.

In [5]:
# Detection function using Hugging Face GroundingDINO
def detect_leaves_grounding_dino(image_path, prompts, box_threshold, text_threshold):
    """Detect leaves using GroundingDINO from Hugging Face."""
    from PIL import Image
    
    image = Image.open(image_path).convert("RGB")
    
    # Prepare inputs
    caption = ". ".join(prompts)
    inputs = grounding_dino_processor(images=image, text=caption, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Run model
    with torch.no_grad():
        outputs = grounding_dino_model(**inputs)
    
    # Post-process outputs
    results = grounding_dino_processor.post_process_grounded_object_detection(
        outputs,
        inputs['input_ids'],
        threshold=box_threshold,
        text_threshold=text_threshold,
        target_sizes=[image.size[::-1]]
    )[0]
    
    # Extract boxes, scores, labels
    boxes = results['boxes'].cpu().numpy()  # Already in xyxy format
    scores = results['scores'].cpu().numpy()
    labels = results['labels']
    
    # Convert image to array for return
    image_array = np.array(image)
    
    return boxes, scores, labels, image_array

# SAM segmentation
def segment_leaf_sam(image_array, box):
    """Segment leaf using SAM."""
    sam_predictor.set_image(image_array)
    masks, scores, _ = sam_predictor.predict(
        box=box,
        multimask_output=True
    )
    best_mask_idx = np.argmax(scores)
    return masks[best_mask_idx], scores[best_mask_idx]

# Validation
def validate_detection(box, mask, image_shape, min_area_ratio):
    """Validate detection quality."""
    h, w = image_shape[:2]
    total_pixels = h * w
    
    box_w = box[2] - box[0]
    box_h = box[3] - box[1]
    box_area = box_w * box_h
    box_area_ratio = box_area / total_pixels
    
    mask_area = np.sum(mask)
    mask_area_ratio = mask_area / total_pixels
    
    is_valid = box_area_ratio >= min_area_ratio
    
    metrics = {
        'box_area_ratio': box_area_ratio,
        'mask_area_ratio': mask_area_ratio,
        'box_width': box_w,
        'box_height': box_h
    }
    
    return is_valid, metrics

# Green dominance check
def check_green_dominance(image_array, mask, threshold=1.2):
    """Check if masked region is predominantly green."""
    if np.sum(mask) == 0:
        return False, 0.0
    
    masked_pixels = image_array[mask]
    R = masked_pixels[:, 0].astype(np.float32)
    G = masked_pixels[:, 1].astype(np.float32)
    B = masked_pixels[:, 2].astype(np.float32)
    
    green_ratio = np.mean(G / ((R + B) / 2 + 1e-6))
    is_green = green_ratio >= threshold
    
    return is_green, float(green_ratio)

# Crop and resize
def crop_and_resize_leaf(image_array, mask, box, target_size=(448, 448), padding_ratio=0.1):
    """Crop leaf region with padding and resize."""
    h, w = image_array.shape[:2]
    x1, y1, x2, y2 = [int(v) for v in box]
    
    # Add padding
    pad_w = int((x2 - x1) * padding_ratio)
    pad_h = int((y2 - y1) * padding_ratio)
    
    x1_pad = max(0, x1 - pad_w)
    y1_pad = max(0, y1 - pad_h)
    x2_pad = min(w, x2 + pad_w)
    y2_pad = min(h, y2 + pad_h)
    
    cropped = image_array[y1_pad:y2_pad, x1_pad:x2_pad]
    cropped_mask = mask[y1_pad:y2_pad, x1_pad:x2_pad]
    
    resized = cv2.resize(cropped, target_size, interpolation=cv2.INTER_LINEAR)
    resized_mask = cv2.resize(cropped_mask.astype(np.uint8), target_size, interpolation=cv2.INTER_NEAREST)
    
    return resized, resized_mask

print("✓ Pipeline helper functions defined (Hugging Face compatible)")

# Fallback: Green channel segmentation
def green_channel_segmentation(image_array, percentile_threshold=60):
    """Segmentation sémantique utilisant dominance du canal vert."""
    R = image_array[:, :, 0].astype(np.float32)
    G = image_array[:, :, 1].astype(np.float32)
    B = image_array[:, :, 2].astype(np.float32)
    
    green_ratio = G / ((R + B) / 2 + 1e-6)
    threshold_value = max(np.percentile(green_ratio, percentile_threshold), 1.05)
    mask = (green_ratio > threshold_value).astype(np.uint8)
    
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)
    
    if num_labels <= 1:
        return None, None
    
    areas = stats[1:, cv2.CC_STAT_AREA]
    largest_idx = 1 + np.argmax(areas)
    
    final_mask = (labels == largest_idx).astype(np.uint8)
    
    x = stats[largest_idx, cv2.CC_STAT_LEFT]
    y = stats[largest_idx, cv2.CC_STAT_TOP]
    w = stats[largest_idx, cv2.CC_STAT_WIDTH]
    h = stats[largest_idx, cv2.CC_STAT_HEIGHT]
    bbox = np.array([x, y, x + w, y + h])
    
    return final_mask, bbox

def fallback_preprocessing(image_array, method='green_semantic',
                          target_size=(448, 448), padding_ratio=0.1):
    """Fallback preprocessing when primary pipeline fails."""
    if method == 'green_semantic':
        mask, bbox = green_channel_segmentation(image_array)
        
        if mask is None or bbox is None:
            method = 'resize'
        else:
            processed, _ = crop_and_resize_leaf(image_array, mask, bbox, target_size, padding_ratio)
            
            metadata = {
                'method': method,
                'bbox': bbox.tolist(),
                'mask_area_ratio': np.sum(mask) / (image_array.shape[0] * image_array.shape[1])
            }
            
            return processed, metadata
    
    if method == 'center_crop':
        h, w = image_array.shape[:2]
        size = min(h, w)
        y1 = (h - size) // 2
        x1 = (w - size) // 2
        cropped = image_array[y1:y1+size, x1:x1+size]
        processed = cv2.resize(cropped, target_size, interpolation=cv2.INTER_LINEAR)
    else:  # resize
        processed = cv2.resize(image_array, target_size, interpolation=cv2.INTER_LINEAR)
    
    metadata = {'method': method, 'bbox': None, 'mask_area_ratio': 1.0}
    return processed, metadata

def preprocess_single_image(image_path,
                           label=None,
                           prompts=PROMPTS,
                           box_threshold=None,
                           text_threshold=TEXT_THRESHOLD,
                           min_area_ratio=None,
                           target_size=TARGET_SIZE,
                           padding_ratio=PADDING_RATIO,
                           enable_fallback=FALLBACK_ENABLED,
                           fallback_method=FALLBACK_METHOD,
                           return_debug_info=False):
    """
    Complete preprocessing pipeline WITH adaptive thresholds and fallback.
    
    Returns:
        success: Boolean
        result: Dictionary with processed image and metadata
    """
    # Apply class-specific thresholds
    if label is not None and label in CLASS_THRESHOLDS:
        class_config = CLASS_THRESHOLDS[label]
        box_threshold = box_threshold or class_config['box_threshold']
        min_area_ratio = min_area_ratio or class_config['min_area_ratio']
    else:
        box_threshold = box_threshold or 0.30
        min_area_ratio = min_area_ratio or 0.025
    
    image = Image.open(image_path).convert("RGB")
    image_array = np.array(image)
    
    # Try PRIMARY pipeline
    try:
        boxes, confidences, labels_detected, _ = detect_leaves_grounding_dino(
            image_path, prompts, box_threshold, text_threshold
        )
        
        if len(boxes) == 0:
            raise ValueError("No leaves detected by GroundingDINO")
        
        best_idx = np.argmax(confidences)
        best_box = boxes[best_idx]
        best_confidence = float(confidences[best_idx])
        best_label = labels_detected[best_idx] if len(labels_detected) > 0 else "leaf"
        
        mask, sam_confidence = segment_leaf_sam(image_array, best_box)
        
        is_valid, validation_metrics = validate_detection(
            best_box, mask, image_array.shape, min_area_ratio
        )
        
        if not is_valid:
            raise ValueError(f"Detection failed validation: {validation_metrics}")
        
        is_green, green_ratio = check_green_dominance(image_array, mask)
        processed_image, processed_mask = crop_and_resize_leaf(
            image_array, mask, best_box, target_size, padding_ratio
        )
        
        result = {
            "processed_image": processed_image,
            "processed_mask": processed_mask,
            "original_size": image_array.shape[:2],
            "box": best_box.tolist(),
            "detection_confidence": best_confidence,
            "sam_confidence": float(sam_confidence),
            "green_ratio": green_ratio,
            "is_green": is_green,
            "detected_label": best_label,
            "metrics": validation_metrics,
            "image_path": str(image_path),
            "method": "primary",
            "used_fallback": False,
            "class_label": label,
            "box_threshold_used": box_threshold,
            "min_area_ratio_used": min_area_ratio
        }
        
        if return_debug_info:
            result["original_image"] = image_array
            result["full_mask"] = mask
        
        return True, result
    
    except Exception as primary_error:
        if not enable_fallback:
            return False, {
                "error": f"Primary failed: {str(primary_error)}",
                "image_path": str(image_path),
                "method": "none",
                "used_fallback": False
            }
        
        # Try FALLBACK
        try:
            processed_image, fallback_metadata = fallback_preprocessing(
                image_array, method=fallback_method, target_size=target_size, padding_ratio=padding_ratio
            )
            
            result = {
                "processed_image": processed_image,
                "processed_mask": None,
                "original_size": image_array.shape[:2],
                "box": fallback_metadata.get('bbox'),
                "detection_confidence": None,
                "sam_confidence": None,
                "green_ratio": None,
                "is_green": None,
                "detected_label": None,
                "metrics": {'box_area_ratio': fallback_metadata.get('mask_area_ratio', 1.0)},
                "image_path": str(image_path),
                "method": f"fallback_{fallback_method}",
                "used_fallback": True,
                "primary_error": str(primary_error),
                "fallback_metadata": fallback_metadata,
                "class_label": label,
                "box_threshold_used": box_threshold,
                "min_area_ratio_used": min_area_ratio
            }
            
            if return_debug_info:
                result["original_image"] = image_array
            
            return True, result
        
        except Exception as fallback_error:
            return False, {
                "error": f"Both failed - Primary: {str(primary_error)}, Fallback: {str(fallback_error)}",
                "image_path": str(image_path),
                "method": "none",
                "used_fallback": False,
                "primary_error": str(primary_error),
                "fallback_error": str(fallback_error)
            }

print("✓ Enhanced preprocessing pipeline defined")

print("✓ Fallback functions defined")


print("  - detect_leaves_grounding_dino()")
print("  - segment_leaf_sam()")
print("  - validate_detection()")
print("  - check_green_dominance()")
print("  - crop_and_resize_leaf()")
print("  - green_channel_segmentation()")
print("  - fallback_preprocessing()")
print("  - preprocess_single_image()")

✓ Pipeline helper functions defined (Hugging Face compatible)
✓ Enhanced preprocessing pipeline defined
✓ Fallback functions defined
  - detect_leaves_grounding_dino()
  - segment_leaf_sam()
  - validate_detection()
  - check_green_dominance()
  - crop_and_resize_leaf()
  - green_channel_segmentation()
  - fallback_preprocessing()
  - preprocess_single_image()


## 6. Checkpointing System

In [6]:
def load_checkpoint(class_label):
    """
    Load checkpoint for a class.
    Returns set of processed image IDs.
    """
    checkpoint_path = OUTPUT_DIR / 'checkpoints' / f'class_{class_label}_checkpoint.json'
    
    if checkpoint_path.exists():
        with open(checkpoint_path, 'r') as f:
            checkpoint = json.load(f)
        processed_ids = set(checkpoint.get('processed_ids', []))
        print(f"  ✓ Loaded checkpoint: {len(processed_ids):,} images already processed")
        return processed_ids, checkpoint.get('stats', {})
    else:
        print(f"  • No checkpoint found, starting fresh")
        return set(), {}

def save_checkpoint(class_label, processed_ids, stats):
    """
    Save checkpoint for a class.
    """
    checkpoint_path = OUTPUT_DIR / 'checkpoints' / f'class_{class_label}_checkpoint.json'
    
    checkpoint = {
        'class_label': class_label,
        'class_name': CLASS_NAMES[class_label],
        'processed_ids': list(processed_ids),
        'total_processed': len(processed_ids),
        'stats': stats,
        'last_updated': datetime.now().isoformat()
    }
    
    with open(checkpoint_path, 'w') as f:
        json.dump(checkpoint, f, indent=2)

def save_processed_image(result, class_label):
    """
    Save processed image and metadata.
    """
    image_id = result['image_id']
    class_name = CLASS_NAMES[class_label]
    
    # Save image
    if 'processed_image' in result and result['processed_image'] is not None:
        img_path = OUTPUT_DIR / 'images' / f"{class_label}_{class_name}" / f"{image_id}.png"
        Image.fromarray(result['processed_image']).save(img_path)
    
    # Save metadata
    meta_path = OUTPUT_DIR / 'metadata' / f"{class_label}_{class_name}" / f"{image_id}.json"
    
    # Remove non-serializable fields
    metadata = result.copy()
    metadata.pop('processed_image', None)
    metadata.pop('processed_mask', None)
    metadata.pop('original_image', None)
    metadata.pop('full_mask', None)
    
    # Helper for numpy types
    def numpy_to_python(obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.bool_):
            return bool(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        return obj
    
    with open(meta_path, 'w') as f:
        json.dump(metadata, f, indent=2, default=numpy_to_python)

print("✓ Checkpointing system defined")

✓ Checkpointing system defined


## 7. Class Processing Function

In [7]:
def process_class(class_label, manifest_df):
    """
    Process all images for a specific class.
    Includes checkpointing and recovery.
    """
    class_name = CLASS_NAMES[class_label]
    
    print(f"\n{'='*70}")
    print(f"PROCESSING CLASS {class_label}: {class_name}")
    print(f"{'='*70}")
    
    # Load checkpoint
    processed_ids, stats = load_checkpoint(class_label)
    
    # Get class data
    class_df = manifest_df[manifest_df['label'] == class_label]
    total_images = len(class_df)
    
    # Filter out already processed
    class_df = class_df[~class_df['image_id'].isin(processed_ids)]
    remaining = len(class_df)
    
    print(f"\nTotal images: {total_images:,}")
    print(f"Already processed: {len(processed_ids):,}")
    print(f"Remaining: {remaining:,}")
    
    if remaining == 0:
        print(f"✓ Class {class_label} already complete!")
        return stats
    
    # Initialize stats
    if not stats:
        stats = {
            'total': 0,
            'successful': 0,
            'failed': 0,
            'primary': 0,
            'fallback': 0,
            'processing_times': []
        }
    
    # Process images
    print(f"\nStarting processing...\n")
    
    for idx, row in tqdm(class_df.iterrows(), total=remaining, desc=f"Class {class_label}"):
        try:
            image_path = Path(row['file_path'])
            image_id = row['image_id']
            
            # Process image (using preprocess_single_image from earlier)
            start_time = time.time()
            success, result = preprocess_single_image(
                image_path,
                label=class_label,
                enable_fallback=FALLBACK_ENABLED,
                return_debug_info=False
            )
            elapsed = time.time() - start_time
            
            # Update result with metadata
            result['image_id'] = image_id
            result['disease'] = row['disease']
            result['source'] = row['source']
            result['processing_time'] = elapsed
            
            # Save if successful
            if success:
                save_processed_image(result, class_label)
                processed_ids.add(image_id)
                
                stats['successful'] += 1
                if result.get('used_fallback', False):
                    stats['fallback'] += 1
                else:
                    stats['primary'] += 1
            else:
                stats['failed'] += 1
            
            stats['total'] += 1
            stats['processing_times'].append(elapsed)
            
            # Checkpoint every N images
            if stats['total'] % CHECKPOINT_INTERVAL == 0:
                save_checkpoint(class_label, processed_ids, stats)
                print(f"  ✓ Checkpoint saved: {stats['total']}/{total_images}")
        
        except Exception as e:
            print(f"  ✗ Error processing {row['image_id']}: {str(e)}")
            stats['failed'] += 1
            stats['total'] += 1
    
    # Final checkpoint
    save_checkpoint(class_label, processed_ids, stats)
    
    # Summary
    print(f"\n{'='*70}")
    print(f"CLASS {class_label} COMPLETE")
    print(f"{'='*70}")
    print(f"Total processed: {stats['total']:,}")
    print(f"Successful: {stats['successful']:,} ({stats['successful']/stats['total']*100:.1f}%)")
    print(f"  Primary: {stats['primary']:,}")
    print(f"  Fallback: {stats['fallback']:,}")
    print(f"Failed: {stats['failed']:,}")
    if stats['processing_times']:
        print(f"Avg time: {np.mean(stats['processing_times'])*1000:.0f} ms/image")
    print(f"{'='*70}\n")
    
    return stats

print("✓ Class processing function defined")

✓ Class processing function defined


## 8. Execute Production Processing

In [8]:
# Process each class sequentially
# (Sequential is better for GPU to avoid memory conflicts)

print("="*70)
print("STARTING PRODUCTION PROCESSING")
print("="*70)
print(f"Total images: {len(manifest_df):,}")
print(f"Classes: {len(CLASS_NAMES)}")
print(f"Checkpoint interval: {CHECKPOINT_INTERVAL:,} images")
print(f"\nEstimated time: {len(manifest_df) * 0.266 / 3600:.1f} hours @ 266 ms/image")
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*70)

# Store results for each class
all_stats = {}
start_time = time.time()

# Process each class
for class_label in sorted(CLASS_NAMES.keys()):
    class_stats = process_class(class_label, manifest_df)
    all_stats[class_label] = class_stats

total_time = time.time() - start_time

print("\n" + "="*70)
print("PRODUCTION PROCESSING COMPLETE")
print("="*70)
print(f"Total time: {total_time/3600:.2f} hours")
print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*70)

STARTING PRODUCTION PROCESSING
Total images: 30,513
Classes: 5
Checkpoint interval: 1,000 images

Estimated time: 2.3 hours @ 266 ms/image
Start time: 2025-12-31 15:00:34

PROCESSING CLASS 0: CBB
  ✓ Loaded checkpoint: 1,087 images already processed

Total images: 1,087
Already processed: 1,087
Remaining: 0
✓ Class 0 already complete!

PROCESSING CLASS 1: CBSD
  ✓ Loaded checkpoint: 5,186 images already processed

Total images: 5,186
Already processed: 5,186
Remaining: 0
✓ Class 1 already complete!

PROCESSING CLASS 2: CGM
  ✓ Loaded checkpoint: 2,386 images already processed

Total images: 2,386
Already processed: 2,386
Remaining: 0
✓ Class 2 already complete!

PROCESSING CLASS 3: CMD
  ✓ Loaded checkpoint: 7,000 images already processed

Total images: 18,158
Already processed: 7,000
Remaining: 11,158

Starting processing...



Class 3:   0%|          | 0/11158 [00:00<?, ?it/s]

/home/hounfodjidagba/miniconda3/envs/cassava/lib/python3.11/site-packages/transformers/models/grounding_dino/processing_grounding_dino.py:93: FutureWarning: The key `labels` is will return integer ids in `GroundingDinoProcessor.post_process_grounded_object_detection` output since v4.51.0. Use `text_labels` instead to retrieve string object names.
  warnings.warn(self.message, FutureWarning)


  ✓ Checkpoint saved: 8000/18158
  ✓ Checkpoint saved: 9000/18158
  ✓ Checkpoint saved: 10000/18158
  ✓ Checkpoint saved: 11000/18158
  ✓ Checkpoint saved: 12000/18158
  ✓ Checkpoint saved: 13000/18158
  ✓ Checkpoint saved: 14000/18158
  ✗ Error processing uuid_02e5df2a-f9ee-4c55-b54b-3b9c809aa7f7.jpg: cannot identify image file '../data/cmd_002/uuid_02e5df2a-f9ee-4c55-b54b-3b9c809aa7f7.jpg'
  ✗ Error processing uuid_1eeaf378-1e19-4d84-b438-37ffc67fc68d.jpg: cannot identify image file '../data/cmd_002/uuid_1eeaf378-1e19-4d84-b438-37ffc67fc68d.jpg'
  ✗ Error processing uuid_2f8e3e94-8741-4ac6-99c8-f488e935202d.jpg: broken data stream when reading image file
  ✗ Error processing uuid_02d522d4-2f34-4bad-b405-bbdd32914c39.jpg: cannot identify image file '../data/cmd_002/uuid_02d522d4-2f34-4bad-b405-bbdd32914c39.jpg'
  ✗ Error processing uuid_2ae5b0a4-ba6b-42f3-9b1e-9bafc225dcd3.jpg: broken data stream when reading image file
  ✓ Checkpoint saved: 15000/18158
  ✗ Error processing uuid_00d85

Class 4:   0%|          | 0/3696 [00:00<?, ?it/s]

  ✓ Checkpoint saved: 1000/3696
  ✓ Checkpoint saved: 2000/3696
  ✓ Checkpoint saved: 3000/3696

CLASS 4 COMPLETE
Total processed: 3,696
Successful: 3,696 (100.0%)
  Primary: 3,577
  Fallback: 119
Failed: 0
Avg time: 284 ms/image


PRODUCTION PROCESSING COMPLETE
Total time: 1.39 hours
End time: 2025-12-31 16:24:04


## 9. Generate Final Summary

In [9]:
# Aggregate statistics
total_stats = {
    'total_images': 0,
    'successful': 0,
    'failed': 0,
    'primary': 0,
    'fallback': 0,
    'by_class': {},
    'processing_time_hours': total_time / 3600,
    'timestamp': datetime.now().isoformat()
}

for class_label, stats in all_stats.items():
    total_stats['total_images'] += stats['total']
    total_stats['successful'] += stats['successful']
    total_stats['failed'] += stats['failed']
    total_stats['primary'] += stats['primary']
    total_stats['fallback'] += stats['fallback']
    
    total_stats['by_class'][class_label] = {
        'name': CLASS_NAMES[class_label],
        'total': stats['total'],
        'successful': stats['successful'],
        'success_rate': stats['successful'] / stats['total'] * 100 if stats['total'] > 0 else 0,
        'primary': stats['primary'],
        'fallback': stats['fallback'],
        'failed': stats['failed'],
        'avg_time_ms': np.mean(stats['processing_times']) * 1000 if stats['processing_times'] else 0
    }

# Save summary
summary_path = OUTPUT_DIR / 'production_summary.json'
with open(summary_path, 'w') as f:
    json.dump(total_stats, f, indent=2)

# Display summary
print("="*70)
print("FINAL SUMMARY")
print("="*70)
print(f"\nTotal Images Processed: {total_stats['total_images']:,}")
print(f"Successful: {total_stats['successful']:,} ({total_stats['successful']/total_stats['total_images']*100:.1f}%)")
print(f"  • Primary pipeline: {total_stats['primary']:,} ({total_stats['primary']/total_stats['total_images']*100:.1f}%)")
print(f"  • Fallback pipeline: {total_stats['fallback']:,} ({total_stats['fallback']/total_stats['total_images']*100:.1f}%)")
print(f"Failed: {total_stats['failed']:,} ({total_stats['failed']/total_stats['total_images']*100:.1f}%)")

print(f"\nProcessing Time: {total_stats['processing_time_hours']:.2f} hours")
print(f"Average: {total_stats['processing_time_hours']*3600*1000/total_stats['total_images']:.0f} ms/image")

print(f"\nBy Class:")
for class_label in sorted(total_stats['by_class'].keys()):
    c = total_stats['by_class'][class_label]
    print(f"  {class_label} - {c['name']:10s}: {c['successful']:,}/{c['total']:,} ({c['success_rate']:.1f}%)")
    print(f"      Primary: {c['primary']:,}, Fallback: {c['fallback']:,}, Failed: {c['failed']}")

print(f"\n✓ Summary saved: {summary_path}")
print("="*70)

if total_stats['successful'] / total_stats['total_images'] >= 0.95:
    print("\n✅ SUCCESS: ≥95% success rate achieved!")
    print("   Dataset ready for VLM training (Phase 5)")
else:
    print(f"\n⚠️  WARNING: Success rate below 95%")
    print(f"   Review failed images and consider adjusting thresholds")

FINAL SUMMARY

Total Images Processed: 30,513
Successful: 30,491 (99.9%)
  • Primary pipeline: 28,995 (95.0%)
  • Fallback pipeline: 1,496 (4.9%)
Failed: 22 (0.1%)

Processing Time: 1.39 hours
Average: 164 ms/image

By Class:
  0 - CBB       : 1,087/1,087 (100.0%)
      Primary: 1,054, Fallback: 33, Failed: 0
  1 - CBSD      : 5,186/5,186 (100.0%)
      Primary: 5,117, Fallback: 69, Failed: 0
  2 - CGM       : 2,386/2,386 (100.0%)
      Primary: 2,342, Fallback: 44, Failed: 0
  3 - CMD       : 18,136/18,158 (99.9%)
      Primary: 16,905, Fallback: 1,231, Failed: 22
  4 - Healthy   : 3,696/3,696 (100.0%)
      Primary: 3,577, Fallback: 119, Failed: 0

✓ Summary saved: /home/hounfodjidagba/learning/cassava/outputs/processed/production_summary.json

✅ SUCCESS: ≥95% success rate achieved!
   Dataset ready for VLM training (Phase 5)
